In [33]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API Key Exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableBranch

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

Bro API Key Exists


# Chain with Conditional Chains

In [34]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["Postive","Negative"]

llm_structure_output = llm_openai.with_structured_output(llm_schema)

##result= llm_structure_output.invoke("The Movie is good")
#result.model_dump()['movie_summary_flag']


In [35]:
# TASK -1[Prompt]


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are Movie Summarizer"),
    ("human","Please summarize the moview in 50 letters: {input}")
])

In [36]:

# TASK-2 [LLM]
llm_structure_output = llm_openai.with_structured_output(llm_schema)

In [37]:
# TASK - 3 - Custom Runnable dict
def pydantic_json(input: llm_schema) -> str:

    return  input.model_dump()['movie_summary_flag']
 
pydantic_json_lambda  = RunnableLambda(pydantic_json) 

# Conditional Chain - 1

In [38]:
# TASK -1 [Promt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Linkedin Post Generator"),
    ("human", " Create a post for the follwing text for Linkedin: {text}")
])

# TASK-2 [llm]

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

#TASK-3 [str output]

str_parser = StrOutputParser()

chain_linekdin = linkedin_prompt | llm_openai | str_parser


# Conditional Chain-2

In [42]:

def insta_chain(text:dict):

    # text = text["text"]
# TASK -1 [prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Instagram Post Generator"),
    ("human", " Create a post for the follwing text for Instagram: {text}")
    ])

    # TASK-2 [llm]
    
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)
    
    #TASK-3 [str output]
    
    str_parser = StrOutputParser()
    
    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke({"text": text})
    return result

insta_chain_runnable = RunnableLambda(insta_chain)

# Final Orchestration

In [43]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linekdin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structure_output | pydantic_json_lambda | conditional_chain

In [44]:
#final_output = final_chain.invoke("Bahubali")
final_orchestrator.invoke({"input": "I love this KGF Movie"})

'Here’s an Instagram-ready post based on “Positive.”\n\nSuggested image\n- Minimalist square (1080x1080) or vertical (1080x1350) graphic.\n- Soft pastel gradient background (sunny yellow → peach) with subtle texture.\n- Centered bold script or handwritten font that reads: “Positive” (large) and a small subline: “Choose it. Live it. Share it.”\n- Small sun or sparkle icon in a corner and plenty of whitespace for a clean look.\n\nCaption (main)\nPositive ✨\nChoose it. Breathe it. Share it.\nDouble-tap if you’re choosing positivity today 💛\n\nTag someone who lifts you up — let’s spread good vibes! ☀️\n\nAlt text (for accessibility)\nGraphic with the word “Positive” in large script on a warm pastel gradient background; small subline reads “Choose it. Live it. Share it.” Sun icon in the corner.\n\nHashtags\n#Positive #PositiveVibes #ChooseJoy #MindsetMatters #Gratitude #GoodVibesOnly #DailyInspo\n\n3 alternative captions (pick one)\n1) Short & sweet: “Positivity is a practice. Start today. 